In [1]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from numba import njit
PROB_COLS = [f'P_state{i}_filtered' for i in range(1, 5)]

In [2]:
import sys
sys.path.insert(0, r"C:\projects\hedge_fund")
from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client
from data.processing.usdinr.features import USDINRFeatureBuilder
import pandas as pd

db = DuckDBClient()
s  = S3Client()
contracts = ["26OCTFUT", "26NOVFUT"]   # or use active_contracts()
frames = []
for contract in contracts:
    files = s.list_files(f"processed/1s/USDINR{contract}/")
    for f in files:
        parts = f.split("/")
        sym  = parts[2]
        date = parts[3].replace(".parquet", "")
        df = db.read_parquet(f"processed/1s/{sym}/{date}.parquet")
        df["symbol"] = sym
        frames.append(df)

db.close()
df_raw = pd.concat(frames, ignore_index=True)
builder = USDINRFeatureBuilder(data=df_raw, auto_build=True)
df = builder.dataset

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (4 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

C:\projects\hedge_fund\data\processing\usdinr\features.py:325: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda day: day["ret_1s"].rolling(300, min_periods=120).corr(day["_ret_lag1"]))


In [3]:
@njit
def _e_step(logb, A, pi, starts, ends):
    T, K = logb.shape
    gamma = np.empty((T, K))
    transitions = np.zeros((K, K))
    pi_counts = np.zeros(K)
    ll = 0.0
    for seq in range(len(starts)):
        lo, hi = starts[seq], ends[seq]
        n = hi-lo
        alpha = np.zeros((n, K))
        beta = np.ones((n, K))
        b = np.empty((n, K))
        scale = np.empty(n)
        for t in range(n):
            peak = np.max(logb[lo+t])
            for k in range(K):
                b[t,k] = np.exp(logb[lo+t,k]-peak)
            for j in range(K):
                if t == 0:
                    alpha[t,j] = pi[j]*b[t,j]
                else:
                    v = 0.0
                    for i in range(K): v += alpha[t-1,i]*A[i,j]
                    alpha[t,j] = b[t,j]*v
            s = max(np.sum(alpha[t]),1e-300)
            scale[t] = s
            for j in range(K): alpha[t,j] /= s
            ll += np.log(s)+peak
        for t in range(n-2,-1,-1):
            for i in range(K):
                v=0.0
                for j in range(K): v += A[i,j]*b[t+1,j]*beta[t+1,j]
                beta[t,i] = v/scale[t+1]
        for t in range(n):
            z=0.0
            for k in range(K):
                gamma[lo+t,k] = alpha[t,k]*beta[t,k]
                z += gamma[lo+t,k]
            for k in range(K): gamma[lo+t,k] /= max(z,1e-300)
        for k in range(K): pi_counts[k] += gamma[lo,k]
        for t in range(n-1):
            for i in range(K):
                for j in range(K):
                    transitions[i,j] += (alpha[t,i]*A[i,j]*b[t+1,j]*beta[t+1,j]/scale[t+1])
    return gamma, transitions, pi_counts, ll

In [4]:
def _log_emissions(counts, returns, q, p, sigma):
    if np.any((counts==0)&(returns!=0)):
        raise ValueError('count=0 with nonzero mid-price return; inspect forward-filling')
    zero = returns==0
    nonzero = ~zero
    out = np.empty((len(counts),len(sigma)))
    for k in range(len(sigma)):
        out[:,k] = np.log(np.maximum(q[k,counts],1e-300))
        out[zero,k] += np.log(np.maximum(p[k,counts[zero]],1e-300))
        x = returns[nonzero]
        out[nonzero,k] += (
            np.log(np.maximum(1-p[k,counts[nonzero]],1e-300))
            -np.log(sigma[k])-0.5*np.log(2*np.pi)-0.5*(x/sigma[k])**2
        )
    return out

In [5]:
def _sequence_arrays(frame):
    lengths = frame.groupby(['symbol','date'],sort=False).size().to_numpy()
    ends = np.cumsum(lengths).astype(np.int64)
    starts = np.r_[0,ends[:-1]].astype(np.int64)
    return (frame['count_cat'].to_numpy(np.int64),
            frame['x_bp'].to_numpy(np.float64),starts,ends)


In [6]:
def _fit_once(counts, returns, starts, ends, K, seed, max_iter, tol):
    rng = np.random.default_rng(seed)
    q = rng.dirichlet(np.full(4,2.0),size=K)
    centers = np.linspace(0,3,K).astype(int)
    for k,c in enumerate(centers):
        q[k,c] += 2
        q[k] /= q[k].sum()
    p = np.empty((K,4))
    p[:,0] = 1
    raw_p = np.array([1]+[
        np.mean(returns[counts==c]==0) if np.any(counts==c) else .5
        for c in (1,2,3)])
    p[:,1:] = np.clip(raw_p[1:]+rng.normal(0,.09,(K,3)),.01,.999)
    nz=returns[returns!=0]
    sigma = max(float(np.std(nz)),.03)*np.exp(rng.normal(0,.35,K))
    A=np.full((K,K),.12/(K-1))
    np.fill_diagonal(A,.88)
    pi=np.full(K,1/K)
    history=[]
    for _ in range(max_iter):
        logb=_log_emissions(counts,returns,q,p,sigma)
        gamma,trans,pi_counts,ll=_e_step(logb,A,pi,starts,ends)
        history.append(ll)
        if len(history)>1 and abs(history[-1]-history[-2])<tol*(1+abs(ll)):
            break
        weights=gamma.sum(axis=0)
        nzmask=returns!=0
        for k in range(K):
            cat_weights=np.bincount(counts,weights=gamma[:,k],minlength=4)
            q[k]=(cat_weights+.01)/(weights[k]+.04)
            for c in (1,2,3):
                zero_weight=gamma[(counts==c)&(returns==0),k].sum()
                p[k,c]=(zero_weight+.1)/(cat_weights[c]+.2)
            p[k,0]=1.0
            w=gamma[nzmask,k]
            sigma[k]=np.sqrt(max(np.sum(w*returns[nzmask]**2)/max(w.sum(),1e-12),1e-8))
        A=(trans+1e-3)/(trans.sum(axis=1,keepdims=True)+K*1e-3)
        pi=(pi_counts+1e-3)/(pi_counts.sum()+K*1e-3)
    final_ll=_e_step(_log_emissions(counts,returns,q,p,sigma),A,pi,starts,ends)[-1]
    return dict(q=q,p=p,sigma=sigma,A=A,pi=pi,ll=final_ll,
                iterations=len(history),history=history)

In [7]:
def _filter(counts, returns, starts, ends, model):
    logb=_log_emissions(counts,returns,model['q'],model['p'],model['sigma'])
    K=len(model['sigma'])
    out=np.empty((len(counts),K))
    for lo,hi in zip(starts,ends):
        for t in range(lo,hi):
            prior=model['pi'] if t==lo else out[t-1]@model['A']
            z=np.log(np.maximum(prior,1e-300))+logb[t]
            out[t]=np.exp(z-logsumexp(z))
    return out


In [8]:
def add_soft_hmm_features(df, train_end_date, K=4, restarts=3,
                          max_iter=70, tol=1e-6, seed=42):
    """Return (df_with_soft_features, fitted_hmm).

    train_end_date is INCLUSIVE. Set it to the last date used to train
    your LightGBM model (never the validation/test dates).
    The HMM itself uses training observations through that date, and
    the state probabilities are filtered causally within each day.
    """
    required=['symbol','date','ts_sec','tick_count','bid_p1','ask_p1']
    missing=[c for c in required if c not in df.columns]
    if missing: raise ValueError(f'Missing columns: {missing}')
    d=df.copy()
    d['_orig_order_']=np.arange(len(d))
    d['date']=pd.to_datetime(d['date']).dt.date
    d=d.sort_values(['symbol','date','ts_sec']).reset_index(drop=True)
    if d.duplicated(['symbol','date','ts_sec']).any():
        raise ValueError('Duplicate contract/date/second timestamps')
    delta=d.groupby(['symbol','date'],sort=False)['ts_sec'].diff().dropna()
    if not delta.eq(1).all():
        raise ValueError('Expected a complete 1-second grid within each session')
    valid_book=(d['bid_p1']>0)&(d['ask_p1']>=d['bid_p1'])&d['ask_p1'].notna()
    if not valid_book.all(): raise ValueError('Missing/invalid bid-ask prices')
    ticks=d['tick_count']
    if ticks.isna().any() or (ticks<0).any() or not np.allclose(ticks%1,0):
        raise ValueError('tick_count must contain nonnegative integer snapshot counts')
    d['mid']=(d['bid_p1']+d['ask_p1'])/2
    d['x_bp']=d.groupby(['symbol','date'],sort=False)['mid'].transform(
        lambda s: 10000*np.log(s).diff())
    d['count_cat']=np.minimum(ticks,3).astype(np.int64)
    violated=(d['count_cat'].eq(0)&d['x_bp'].notna()&d['x_bp'].ne(0))
    if violated.any():
        raise ValueError(f'{violated.sum()} zero-count nonzero-return observations')
    # First row of each daily sequence lacks a previous price; don't treat as zero return.
    usable=d[d['x_bp'].notna()].copy()
    cutoff=pd.Timestamp(train_end_date).date()
    train=usable[usable['date']<=cutoff].copy()
    future=usable[usable['date']>cutoff]
    if train.empty or future.empty:
        raise ValueError('train_end_date must yield nonempty past and future periods')
    if train['date'].nunique()<3:
        raise ValueError('Need at least three training dates')
    train_counts,train_returns,starts,ends=_sequence_arrays(train)
    best=None
    for r in range(restarts):
        fitted=_fit_once(train_counts,train_returns,starts,ends,K,seed+r,max_iter,tol)
        if best is None or fitted['ll']>best['ll']:
            best=fitted
    # Label states by descending probability of count>=1, for reproducibility.
    activity=1-best['q'][:,0]
    order=np.argsort(-activity)
    for key in ('q','p','sigma','pi'):
        best[key]=best[key][order]
    best['A']=best['A'][np.ix_(order,order)]
    counts,returns,starts,ends=_sequence_arrays(usable)
    probs=_filter(counts,returns,starts,ends,best)
    for j,col in enumerate(PROB_COLS): d[col]=np.nan
    d.loc[usable.index,PROB_COLS]=probs
    d['state_filtered']=np.nan
    d['confidence_filtered']=np.nan
    d.loc[usable.index,'state_filtered']=np.argmax(probs,axis=1)+1
    d.loc[usable.index,'confidence_filtered']=np.max(probs,axis=1)
    if not np.allclose(probs.sum(axis=1),1,atol=1e-8):
        raise RuntimeError('Posterior probabilities not normalized')
    d=d.sort_values('_orig_order_').drop(columns=['_orig_order_'])
    return d,best


In [9]:
# Your dataset should already be loaded as df

df_hmm, hmm_model = add_soft_hmm_features(
    df=df,
    train_end_date="2026-10-31",  # Change to your actual training cutoff
    K=4,
    restarts=3,
    max_iter=70,
    tol=1e-6,
    seed=42
)

# Display generated probabilities
hmm_cols = [
    "P_state1_filtered",
    "P_state2_filtered",
    "P_state3_filtered",
    "P_state4_filtered"
]

print(df_hmm[hmm_cols].head(20))

# Display estimated transition matrix
print("\nTransition matrix:")
print(hmm_model["A"])

# Save the enriched dataset
df_hmm.to_parquet("usdinr_with_soft_hmm.parquet", index=False)

    P_state1_filtered  P_state2_filtered  P_state3_filtered  P_state4_filtered
0                 NaN                NaN                NaN                NaN
1        4.217378e-02       4.869943e-02           0.899600       9.526730e-03
2        1.661785e-02       7.322326e-02           0.903635       6.523669e-03
3        9.819256e-02       5.749277e-02           0.834858       9.457052e-03
4        5.787288e-08       2.993589e-14           1.000000       1.178929e-14
5        7.728038e-03       8.835011e-03           0.978402       5.035068e-03
6        8.023008e-02       1.123586e-02           0.898964       9.569849e-03
7        1.935897e-01       1.241877e-02           0.784140       9.851556e-03
8        3.468091e-01       1.240241e-02           0.632360       8.428882e-03
9        5.172175e-01       1.147590e-02           0.464886       6.420331e-03
10       6.700871e-01       1.018672e-02           0.315196       4.529982e-03
11       2.765597e-01       4.961448e-02           0